RA1 - Data Ingestion and Data Quality

In [ ]:
#1.Conectando com a fonte de dados do Kaggle

# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np

file_path = "diabetic_data.csv"

# Carregando versão mais atualizada
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "razanihababdellatif/diabetes-hospital-readmission-dataset",
  file_path,
)

print("Primeiros registros:", df.head())

In [ ]:
#2.⁠ ⁠Inspecionar o dataset e identificar nulos/tipos

# Inspecionar informações gerais, tipos de dados e nulos aparentes
df.info()

# O dataset usa '?' para representar dados faltantes. Vamos padronizar para NaN:
df.replace('?', np.nan, inplace=True)

# Identificar valores ausentes reais por coluna
missing_values = df.isnull().sum()
print("\nValores ausentes por coluna:\n", missing_values[missing_values > 0])

In [ ]:
#3.⁠ ⁠Tratar registros duplicados

# Verificar se existem regis tros duplicados com base no 'encounter_id' (ID do atendimento)
duplicados_encounter = df['encounter_id'].duplicated().sum()
print(f"Total de encounter_ids duplicados: {duplicados_encounter}")

# Verificar duplicatas exatas na linha inteira
duplicados_linhas = df.duplicated().sum()
print(f"Total de linhas totalmente duplicadas: {duplicados_linhas}")

# Remover linhas totalmente duplicadas, caso existam
df.drop_duplicates(inplace=True)

In [ ]:
#Retirar colunas irrelevantes para a análise (muitos valores ausentes ou não informativas)
colunas_para_remover = [
    'weight',
    'payer_code',
    'medical_specialty',
    'max_glu_serum',
    'A1Cresult'
    
]

df.drop(
    columns=colunas_para_remover,
    inplace=True,
    errors='ignore'
)

print("Colunas restantes:", df.shape[1])
print([
    col for col in colunas_para_remover
    if col in df.columns
])

In [ ]:
duplicados_encounter = df['encounter_id'].duplicated().sum()
duplicados_linhas = df.duplicated().sum()

print("IDs de atendimento duplicados:",
      duplicados_encounter)
print("Linhas totalmente duplicadas:",
      duplicados_linhas)

df.drop_duplicates(inplace=True)

In [ ]:
df['encounter_id'] = df['encounter_id'].astype(str)
df['patient_nbr'] = df['patient_nbr'].astype(str)
df.info()

## RA2 - Data Stream

1. Objetivo da análise

O objetivo desta etapa é demonstrar a utilização do Apache Spark Structured Streaming para processar novos registros de atendimentos hospitalares relacionados ao projeto Readmission Rates.

Como a base utilizada no RA1 é estática, será simulada a chegada de registros artificiais em diferentes momentos. A análise utilizará janelas temporais para contabilizar o número de atendimentos e de readmissões em até 30 dias após a alta hospitalar.

A implementação permitirá observar como os resultados são atualizados conforme novos registros são recebidos pelo sistema de streaming.

In [5]:
import os

java_home = (
    "/opt/homebrew/Cellar/openjdk@17/17.0.17/"
    "libexec/openjdk.jdk/Contents/Home"
)

os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = java_home + "/bin:" + os.environ["PATH"]

print("JAVA_HOME:", os.environ["JAVA_HOME"])

JAVA_HOME: /opt/homebrew/Cellar/openjdk@17/17.0.17/libexec/openjdk.jdk/Contents/Home


In [6]:
#Importando bibliotecas necessárias para o streaming

import json
import time
import shutil
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType
)
from pyspark.sql.functions import (
    col,
    window,
    count,
    sum as spark_sum
)

print("Bibliotecas carregadas.")

Bibliotecas carregadas.


In [7]:
#preparando diretórios para streaming
base_dir = Path("streaming_demo")
input_dir = base_dir / "input"
checkpoint_dir = base_dir / "checkpoint"

# Limpar arquivos e checkpoints de execuções anteriores
if base_dir.exists():
    shutil.rmtree(base_dir)

input_dir.mkdir(parents=True)

print("Pasta de entrada:", input_dir.resolve())
print("Checkpoint:", checkpoint_dir.resolve())

Pasta de entrada: /Users/leticiafabri/Desktop/Data-analytics/streaming_demo/input
Checkpoint: /Users/leticiafabri/Desktop/Data-analytics/streaming_demo/checkpoint


In [ ]:
#iniciando sessão Spark
spark = (
    SparkSession.builder
    .appName("ReadmissionRatesStreaming")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Sessão Spark iniciada com sucesso.")

Spark version: 4.1.2
Sessão Spark iniciada com sucesso.


26/10/14 17:57:53 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [9]:
#definindo schema para o streaming
#O campo readmitted_30_days vale 1 quando o atendimento simulado representa uma readmissão em até 30 dias e 0 quando não representa.

schema = StructType([
    StructField("encounter_id", StringType(), False),
    StructField("event_time", TimestampType(), False),
    StructField("time_in_hospital", IntegerType(), False),
    StructField("readmitted_30_days", IntegerType(), False)
])

print("Esquema definido.")

Esquema definido.


## Janelas temporais

Será utilizada uma janela temporal tumbling com duração de 10 segundos. Esse tipo de janela divide a linha do tempo em intervalos fixos e não sobrepostos.

Em cada janela, será calculado o total de atendimentos recebidos e a soma dos indicadores de readmissão em até 30 dias. O processamento utilizará o modo update, que permite exibir as atualizações das agregações à medida que novos registros são processados.

Será configurado um watermark de um minuto para permitir o tratamento de eventos que cheguem atrasados.

In [10]:
#lendo o fluxo e agregando por janela de tempo de 10 segundos, com watermark de 1 minuto para lidar com atrasos na chegada dos dados.

stream_df = (
    spark.readStream
    .schema(schema)
    .option("maxFilesPerTrigger", 1)
    .json(str(input_dir.resolve()))
)

aggregated_df = (
    stream_df
    .withWatermark("event_time", "1 minute")
    .groupBy(
        window(col("event_time"), "10 seconds")
    )
    .agg(
        count("*").alias("total_atendimentos"),
        spark_sum("readmitted_30_days").alias(
            "total_readmissoes_30_dias"
        )
    )
    .select(
        col("window.start").alias("inicio_janela"),
        col("window.end").alias("fim_janela"),
        col("total_atendimentos"),
        col("total_readmissoes_30_dias")
    )
)

print("Fluxo e agregação configurados.")

Fluxo e agregação configurados.


In [11]:
#iniciando o streaming e exibindo os resultados no console a cada 5 segundos.
query = (
    aggregated_df.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", str(checkpoint_dir.resolve()))
    .trigger(processingTime="5 seconds")
    .start()
)

print("Streaming iniciado:", query.isActive)

Streaming iniciado: True


26/10/14 17:58:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/10/14 17:58:07 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


In [12]:
#criar os registros simulados para o streaming. Cada registro representa um atendimento hospitalar com informações sobre readmissão em até 30 dias.
registros = [
    {
        "encounter_id": "SIM001",
        "event_time": "2026-10-14T15:00:01",
        "time_in_hospital": 4,
        "readmitted_30_days": 1
    },
    {
        "encounter_id": "SIM002",
        "event_time": "2026-10-14T15:00:05",
        "time_in_hospital": 2,
        "readmitted_30_days": 0
    },
    {
        "encounter_id": "SIM003",
        "event_time": "2026-10-14T15:00:12",
        "time_in_hospital": 6,
        "readmitted_30_days": 1
    },
    {
        "encounter_id": "SIM004",
        "event_time": "2026-10-14T15:00:16",
        "time_in_hospital": 3,
        "readmitted_30_days": 0
    },
    {
        "encounter_id": "SIM005",
        "event_time": "2026-10-14T15:00:22",
        "time_in_hospital": 5,
        "readmitted_30_days": 1
    }
]

print("Total de registros simulados:", len(registros))

Total de registros simulados: 5


In [13]:
#enviar os registros simulados para o diretório de entrada, um a cada 3 segundos, para simular a chegada de dados em tempo real.
for i, registro in enumerate(registros, start=1):
    arquivo = input_dir / f"registro_{i:03d}.json"

    with open(arquivo, "w", encoding="utf-8") as f:
        json.dump(registro, f)

    print(f"Novo registro enviado: {registro['encounter_id']}")

    time.sleep(3)

Novo registro enviado: SIM001
Novo registro enviado: SIM002
-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------------------+------------------+-------------------------+
|inicio_janela      |fim_janela         |total_atendimentos|total_readmissoes_30_dias|
+-------------------+-------------------+------------------+-------------------------+
|2026-10-14 15:00:00|2026-10-14 15:00:10|1                 |1                        |
+-------------------+-------------------+------------------+-------------------------+

Novo registro enviado: SIM003
-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+-------------------+------------------+-------------------------+
|inicio_janela      |fim_janela         |total_atendimentos|total_readmissoes_30_dias|
+-------------------+-------------------+------------------+-------------------------+
|2026-10-14 15:00:0

-------------------------------------------
Batch: 3
-------------------------------------------
+-------------------+-------------------+------------------+-------------------------+
|inicio_janela      |fim_janela         |total_atendimentos|total_readmissoes_30_dias|
+-------------------+-------------------+------------------+-------------------------+
|2026-10-14 15:00:10|2026-10-14 15:00:20|2                 |1                        |
+-------------------+-------------------+------------------+-------------------------+

-------------------------------------------
Batch: 4
-------------------------------------------
+-------------------+-------------------+------------------+-------------------------+
|inicio_janela      |fim_janela         |total_atendimentos|total_readmissoes_30_dias|
+-------------------+-------------------+------------------+-------------------------+
|2026-10-14 15:00:20|2026-10-14 15:00:30|1                 |1                        |
+-------------------+-

In [14]:
# Aguarda o processamento dos arquivos disponíveis
query.processAllAvailable()

print("Streaming ativo antes do encerramento:", query.isActive)

# Encerra a consulta de forma controlada
query.stop()
query.awaitTermination()

spark.stop()

print("Streaming encerrado.")

Streaming ativo antes do encerramento: True


26/10/14 17:59:05 WARN DAGScheduler: Failed to cancel job group 6e174d33-b900-4e1e-86d0-d13120821416. Cannot find active jobs for it.
26/10/14 17:59:05 WARN DAGScheduler: Failed to cancel job group 6e174d33-b900-4e1e-86d0-d13120821416. Cannot find active jobs for it.


Streaming encerrado.


26/10/14 17:59:17 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:1440)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:1386)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:1079)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:539)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1136)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:635)
	at java.base/java.lang.Thre

## Resultados e interpretação

Durante a execução, o Spark Structured Streaming recebeu registros artificiais por meio de arquivos JSON e realizou agregações utilizando janelas temporais *tumbling* de 10 segundos.

Os resultados exibidos no console demonstraram atualizações progressivas nas contagens de atendimentos e readmissões simuladas. Como os registros possuem horários diferentes, eles são distribuídos entre janelas temporais distintas.

O modo `update` permite observar as alterações nas agregações à medida que novos registros são processados. O experimento demonstra, portanto, o recebimento de novos dados, o processamento em streaming e a aplicação de janelas temporais.

Os resultados são exclusivamente demonstrativos, pois os registros foram criados artificialmente e não representam estatísticas reais de readmissão hospitalar.

## Conclusão

A atividade demonstrou a utilização do Apache Spark Structured Streaming para processar dados simulados de atendimentos hospitalares. A criação de arquivos em momentos diferentes permitiu representar a chegada contínua de novos registros.

A agregação por janelas temporais de 10 segundos possibilitou contabilizar os atendimentos e as readmissões simuladas em cada intervalo. Dessa forma, a implementação atendeu ao objetivo de demonstrar o processamento em streaming e a utilização de janelas temporais no contexto do projeto Readmission Rates.